In [18]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

###### Completion Model 

In [ ]:
from langchain_openai import OpenAI

llmModel = OpenAI()

###### Chat Completion Model

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-instruct")


###### ###### Prompts and Prompt Templates

In [36]:
## This is for completion model.
from langchain_core.prompts import PromptTemplate

Prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} small story about place {topic}"
)

llmModelPrompt = Prompt_template.format(
    adjective = "story",
    topic = "Banglore"
)

response = chatModel.invoke(llmModelPrompt)
print(response)

content="Once upon a time in Bangalore, a bustling city known for its vibrant culture and technological advancements, there lived a young girl named Maya. Maya was a software engineer who worked for a leading tech company in the city.\n\nEvery day, Maya would wake up early in the morning and make her way to work, navigating through the busy streets of Bangalore. She would pass by colorful markets selling fresh fruits and vegetables, and smell the delicious aromas of street food being cooked.\n\nDespite her hectic work schedule, Maya always found time to explore the city and its many attractions. She would visit the beautiful Lalbagh Botanical Garden, stroll through the historic Cubbon Park, and shop at the trendy boutiques in Indiranagar.\n\nBut what Maya loved most about Bangalore was its people. She made friends from all walks of life, from the tech-savvy professionals to the traditional artisans. They would gather at quaint cafes and lively pubs, sharing stories and laughter late in

## This is for Chat completion model.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
    ("system", "You are an {Profession} expert on {topic}."),
    ("human", "Hello, Mr. {Profession}, can you please answer a question?"),
    ("ai", "Sure!"),
    ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    Profession= "Historian",
    topic="The kennedy family",
    user_input="How many grandchildren had Joshep P. Kennedy? "
)

response = chatModel.invoke(messages)

print(response)

content='Joseph P. Kennedy, Sr., the patriarch of the Kennedy family, had a total of 34 grandchildren. These grandchildren are from his nine children: Joseph Jr., John F. Kennedy, Rosemary, Kathleen, Eunice, Patricia, Robert F. Kennedy, Jean, and Ted Kennedy.' response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 59, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-d66fd4d4-527c-4cb4-bda3-aa150ecd0677-0' usage_metadata={'input_tokens': 59, 'output_tokens': 60, 'total_tokens': 119}


###### Few Shot Promting

In [16]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(

[
    ("human", "{input}"),
    ("ai", "{output}"),
]
)

few_shot_prompt= FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator"),
        few_shot_prompt,
        ("human", "{input}"), 
    ]
)

chain = final_prompt | chatModel

response = chain.invoke({"input": "How are you?"})

print(response.content)

##print(response.content) provides full response with metadata


¿Cómo estás?


In [17]:
chatModel.invoke("How to makdown cells in jupitor note book").content

'To create markdown cells in Jupyter Notebook, follow these steps:\n\n1. Open Jupyter Notebook and create a new notebook or open an existing one.\n\n2. Click on the "+" button in the toolbar to add a new cell.\n\n3. In the cell type "Esc" to enter command mode and then press "M" to change the cell type to Markdown.\n\n4. You can now type your markdown content in the cell. Markdown is a lightweight markup language with plain text formatting syntax that can be converted to HTML.\n\n5. To render the markdown content, press "Shift + Enter" or click the "Run" button in the toolbar.\n\n6. You can also use markdown syntax to format text, add headers, lists, links, images, etc. Here are a few examples:\n\n- To create a header, use "# Header 1" for the largest header, "## Header 2" for the second largest header, and so on.\n- To create a list, use "- Item 1" to create a bullet point list or "1. Item 1" to create a numbered list.\n- To add a link, use "[link text](http://www.example.com)".\n- To

## Output Parser

In [20]:
from langchain_openai import OpenAI

llmModel = OpenAI()

In [19]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [33]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# Import the modern ChatOpenAI class

# 2. Build your prompt template
json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

# 3. Initialize the parser
json_parser = JsonOutputParser()

# 4. Assemble the LCEL chain
json_chain = json_prompt | chatModel | json_parser


In [34]:
json_chain.invoke({"question" : "What is the biggest country"})

{'answer': 'The biggest country in the world is Russia.'}

###### Optionally, We can use Pydantic to define a custom output format.

In [37]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

# Define a Pydantic Object with the desired output format.

class Joke(BaseModel):
    setup : str = Field(description="question to setp a joke")
    punchline : str = Field(description="answer to resolve the joke")

In [38]:
# defne the parser refering the Pydantic Object.

parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser formate instruction in the prompt definition.
prompt = PromptTemplate(
    template= "Answer the user query. \n{format_instruction}\n{query}\n",
    input_variables = ["query"],
    partial_variables={"format_instruction": parser.get_format_instructions()},
)

chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke"})

{'setup': "Why couldn't the bicycle find its way home?",
 'punchline': 'Because it lost its bearings!'}